In [1]:
import pandas as pd
import numpy as np                  
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("./train.csv")
print(df.head())
print(df.shape)
print(df.isnull().sum())

   id  Gender  Age  Driving_License  Region_Code  Previously_Insured  \
0   1    Male   44                1         28.0                   0   
1   2    Male   76                1          3.0                   0   
2   3    Male   47                1         28.0                   0   
3   4    Male   21                1         11.0                   1   
4   5  Female   29                1         41.0                   1   

  Vehicle_Age Vehicle_Damage  Annual_Premium  Policy_Sales_Channel  Vintage  \
0   > 2 Years            Yes         40454.0                  26.0      217   
1    1-2 Year             No         33536.0                  26.0      183   
2   > 2 Years            Yes         38294.0                  26.0       27   
3    < 1 Year             No         28619.0                 152.0      203   
4    < 1 Year             No         27496.0                 152.0       39   

   Response  
0         1  
1         0  
2         1  
3         0  
4         0  
(381109,

In [3]:
x = df.drop(["Response","id"],axis = 1)
y = df["Response"]

In [4]:
from sklearn.model_selection import train_test_split

x_train,x_temp,y_train,y_temp = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

x_val,x_test,y_val,y_test = train_test_split(x_temp,y_temp,test_size=0.5,random_state=42,stratify=y_temp)

In [5]:
print(x_train.dtypes)

Gender                   object
Age                       int64
Driving_License           int64
Region_Code             float64
Previously_Insured        int64
Vehicle_Age              object
Vehicle_Damage           object
Annual_Premium          float64
Policy_Sales_Channel    float64
Vintage                   int64
dtype: object


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,FunctionTransformer
from sklearn.pipeline import Pipeline

numeric_cols = [
    "Age",
    "Driving_License",
    "Previously_Insured",
    "Vintage"
]
categorical_cols = [
    "Gender",
    "Vehicle_Damage",
    "Region_Code",
    "Policy_Sales_Channel",
    "Vehicle_Age"
]
premium_col = [
    "Annual_Premium"
]

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown = "ignore")
premium_transformer = Pipeline([
    ("log", FunctionTransformer(np.log1p)),
    ("scaler", StandardScaler())
])

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("premium", premium_transformer, premium_col)
    ]
)
preprocessor.fit(x_train)

x_train_processed = preprocessor.transform(x_train)

x_val_processed = preprocessor.transform(x_val)

x_test_processed = preprocessor.transform(x_test)

In [8]:
print(x_train_processed.shape)
print(x_train_processed[:2])

(304887, 219)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20 stored elements and shape (2, 219)>
  Coords	Values
  (0, 0)	-1.0845375320075197
  (0, 1)	0.04636451425338886
  (0, 2)	-0.9194842270922522
  (0, 3)	0.5809451938819227
  (0, 5)	1.0
  (0, 7)	1.0
  (0, 48)	1.0
  (0, 205)	1.0
  (0, 216)	1.0
  (0, 218)	0.128782764267634
  (1, 0)	-0.8267270604047354
  (1, 1)	0.04636451425338886
  (1, 2)	1.0875662360869076
  (1, 3)	-1.3074218455910156
  (1, 4)	1.0
  (1, 6)	1.0
  (1, 16)	1.0
  (1, 85)	1.0
  (1, 216)	1.0
  (1, 218)	0.5374335523817526


In [9]:
print(type(x_train_processed))
print(x_train_processed.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(304887, 219)


In [10]:
x_train_processed = x_train_processed.toarray()
x_val_processed = x_val_processed.toarray()
x_test_processed = x_test_processed.toarray()

In [11]:
print(x_train_processed.shape)

(304887, 219)


In [12]:
input_dim = x_train_processed.shape[1]

In [16]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,models

model = keras.Sequential([
    layers.Input(shape = (input_dim)),
    
    layers.Dense(128, activation = "relu"),
    layers.Dense(64,activation="relu"),
    layers.Dense(1,activation = 'sigmoid')
])

model.compile(
    optimizer = 'adam',
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

model.summary()



Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 128)               28160     
                                                                 
 dense_10 (Dense)            (None, 64)                8256      
                                                                 
 dense_11 (Dense)            (None, 1)                 65        
                                                                 
Total params: 36,481
Trainable params: 36,481
Non-trainable params: 0
_________________________________________________________________


In [17]:
history = model.fit(
    x_train_processed,
    y_train,
    validation_data = (x_val_processed,y_val),
    epochs=10,
    batch_size=1024
)

Epoch 1/10
298/298 [==============================] - 13s 10ms/step - loss: 0.2848 - accuracy: 0.8760 - val_loss: 0.2650 - val_accuracy: 0.8775
Epoch 2/10
298/298 [==============================] - 2s 8ms/step - loss: 0.2651 - accuracy: 0.8775 - val_loss: 0.2647 - val_accuracy: 0.8775
Epoch 3/10
298/298 [==============================] - 2s 8ms/step - loss: 0.2642 - accuracy: 0.8775 - val_loss: 0.2655 - val_accuracy: 0.8775
Epoch 4/10
298/298 [==============================] - 2s 7ms/step - loss: 0.2635 - accuracy: 0.8777 - val_loss: 0.2649 - val_accuracy: 0.8776
Epoch 5/10
298/298 [==============================] - 2s 7ms/step - loss: 0.2630 - accuracy: 0.8777 - val_loss: 0.2646 - val_accuracy: 0.8774
Epoch 6/10
298/298 [==============================] - 2s 8ms/step - loss: 0.2625 - accuracy: 0.8778 - val_loss: 0.2641 - val_accuracy: 0.8776
Epoch 7/10
298/298 [==============================] - 2s 8ms/step - loss: 0.2620 - accuracy: 0.8778 - val_loss: 0.2656 - val_accuracy: 0.8776
Epoc

In [19]:
test_loss,test_accuracy = model.evaluate(
    x_test_processed,
    y_test
)
print("test loss", test_loss)
print("test accuracy", test_accuracy)

1191/1191 [==============================] - 7s 6ms/step - loss: 0.2665 - accuracy: 0.8767
test loss 0.2665112018585205
test accuracy 0.8767022490501404
